In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

import time

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

ESTRATEGIA_DE_REBALANCEO = "NearMiss_SMOTE"
MODELO = "mlp"

NOMBRE_EXPERIMENTO = f"BCCC17__split__v1__{ESTRATEGIA_DE_REBALANCEO}_pca4_{MODELO}__v1"
CARPETA_DATASET = "BCCC17__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG MLP =====
MLP_HIDDEN_LAYER_SIZES = (100,)
MLP_ACTIVATION = "relu"
MLP_SOLVER = "adam"
MLP_ALPHA = 0.0001
MLP_BATCH_SIZE = "auto"
MLP_LEARNING_RATE = "adaptive"
MLP_LEARNING_RATE_INIT = 0.001
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_VALIDATION_FRACTION = 0.1
MLP_N_ITER_NO_CHANGE = 10
MLP_VERBOSE = False

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 1

# ===== CONFIG REBALANCEO DENTRO DEL CV =====
TARGET_N = 10000
NEARMISS_VERSION = 1
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS = 3

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(878920, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,80,0,0.000000,1,0,0,0,0.000000,0.000000e+00,0.000000,...,0,0,0,1.499262e+09,1.499262e+09,0.000000,0.000000,0.000000,0.000000,1
1,53,1,0.062127,4,64,55,32,43.500000,1.322500e+02,0.000000,...,0,0,0,2.070904e-02,9.500000e-07,0.000000,0.000003,0.000000,0.000000,0
2,80,0,9.282073,15,20,11595,0,774.333333,8.363370e+06,36.000000,...,1,0,0,6.630052e-01,1.031341e+00,2.841435,0.045323,10.000000,20.000000,3
3,443,0,0.000000,1,0,0,0,0.000000,0.000000e+00,0.000000,...,0,0,0,1.499098e+09,1.499098e+09,0.000000,0.000000,0.000000,0.000000,0
4,22,0,12.005657,54,2008,976,0,88.018519,3.527883e+04,18226.380165,...,1,0,0,2.265218e-01,5.716980e-01,0.900982,0.387251,3.666667,334.666667,7


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,360000
1,276914
2,129058
3,76583
4,7625
5,6691
6,5485
7,4759
8,4406


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (878920, 63)
Shape y_train: (878920,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
        activation=MLP_ACTIVATION,
        solver=MLP_SOLVER,
        alpha=MLP_ALPHA,
        batch_size=MLP_BATCH_SIZE,
        learning_rate=MLP_LEARNING_RATE,
        learning_rate_init=MLP_LEARNING_RATE_INIT,
        max_iter=MLP_MAX_ITER,
        early_stopping=MLP_EARLY_STOPPING,
        validation_fraction=MLP_VALIDATION_FRACTION,
        n_iter_no_change=MLP_N_ITER_NO_CHANGE,
        verbose=MLP_VERBOSE,
        random_state=RANDOM_STATE
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",1
,"copy copy: bool, default=TrueIf False, data passed to fit are 

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
labels_globales = np.array(sorted(y_train.unique()))

resultados_folds = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):

    print("=" * 80)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 80)

    # =========================
    # Split del fold
    # =========================
    df_train_fold = df_train.iloc[train_idx].copy()
    df_val_fold = df_train.iloc[val_idx].copy()

    print("Shape train fold original:", df_train_fold.shape)
    print("Shape val fold original  :", df_val_fold.shape)
    print()

    # =========================
    # Rebalanceo SOLO sobre train fold
    # =========================
    df_train_fold_balanceado = rebalancear_train_fold(
        df_fold_train=df_train_fold,
        label_col=LABEL_COL,
        target_n=TARGET_N,
        random_state=RANDOM_STATE + fold,
        nearmiss_version=NEARMISS_VERSION,
        smote_k_neighbors=SMOTE_K_NEIGHBORS,
        estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
        enn_n_neighbors=ENN_N_NEIGHBORS
    )

    X_train_fold_bal = df_train_fold_balanceado.drop(columns=[LABEL_COL])
    y_train_fold_bal = df_train_fold_balanceado[LABEL_COL]

    X_val_fold = df_val_fold.drop(columns=[LABEL_COL])
    y_val_fold = df_val_fold[LABEL_COL]

    # =========================
    # Modelo nuevo para cada fold
    # =========================
    pipeline_fold = Pipeline([
        ("scaler", RobustScaler()),
        ("pca", PCA(n_components=N_COMPONENTS_PCA)),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
            activation=MLP_ACTIVATION,
            solver=MLP_SOLVER,
            alpha=MLP_ALPHA,
            batch_size=MLP_BATCH_SIZE,
            learning_rate=MLP_LEARNING_RATE,
            learning_rate_init=MLP_LEARNING_RATE_INIT,
            max_iter=MLP_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            validation_fraction=MLP_VALIDATION_FRACTION,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            verbose=MLP_VERBOSE,
            random_state=RANDOM_STATE + fold
        ))
    ])
    
    # =========================
    # Entrenamiento
    # =========================
    t0 = time.time()
    pipeline_fold.fit(X_train_fold_bal, y_train_fold_bal)
    fit_time = time.time() - t0

    # =========================
    # Validación
    # =========================
    t0 = time.time()
    y_pred_val = pipeline_fold.predict(X_val_fold)
    score_time = time.time() - t0

    roc_auc_val = calcular_roc_auc_multiclase_seguro(
        modelo=pipeline_fold,
        X_val=X_val_fold,
        y_val=y_val_fold,
        labels_globales=labels_globales
    )

    metricas_fold = {
        "fold": fold,

        "train_original_rows": int(df_train_fold.shape[0]),
        "train_balanceado_rows": int(df_train_fold_balanceado.shape[0]),
        "val_rows": int(df_val_fold.shape[0]),

        "accuracy": accuracy_score(y_val_fold, y_pred_val),

        "precision_weighted": precision_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),

        "precision_macro": precision_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),

        "mcc": matthews_corrcoef(y_val_fold, y_pred_val),
        "roc_auc": roc_auc_val,

        "fit_time": fit_time,
        "score_time": score_time
    }

    resultados_folds.append(metricas_fold)

    print("Métricas fold:")
    print(metricas_fold)
    print()

FOLD 1/5


Shape train fold original: (703136, 64)
Shape val fold original  : (175784, 64)

Estrategia de rebalanceo: NearMiss_SMOTE
Distribución antes del rebalanceo:
LABEL
0     288000
1     221531
2     103247
3      61266
4       6100
5       5353
6       4388
7       3807
8       3525
9       3278
10      1748
11       869
12        16
13         8
Name: count, dtype: int64



Distribución después del undersampling:
LABEL
0     10000
1     10000
2     10000
3     10000
4      6100
5      5353
6      4388
7      3807
8      3525
9      3278
10     1748
11      869
12       16
13        8
Name: count, dtype: int64



SMOTE aplicado con k_neighbors=5
Distribución después de SMOTE:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
Name: count, dtype: int64

No se ha aplicado ENN.

Distribución final después del rebalanceo:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
Name: count, dtype: int64
Shape final: (140000, 64)



No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 1, 'train_original_rows': 703136, 'train_balanceado_rows': 140000, 'val_rows': 175784, 'accuracy': 0.08710121512765667, 'precision_weighted': 0.009741943338766589, 'recall_weighted': 0.08710121512765667, 'f1_weighted': 0.017523788954689763, 'precision_macro': 0.00798772601500296, 'recall_macro': 0.0952054517700532, 'f1_macro': 0.014368675810316369, 'mcc': 0.03904669739631608, 'roc_auc': nan, 'fit_time': 17.94218111038208, 'score_time': 0.13856959342956543}

FOLD 2/5


...

No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 4, 'train_original_rows': 703136, 'train_balanceado_rows': 140000, 'val_rows': 175784, 'accuracy': 0.31515382514904655, 'precision_weighted': 0.16515972149117308, 'recall_weighted': 0.31515382514904655, 'f1_weighted': 0.21646821885037457, 'precision_macro': 0.029185234774055928, 'recall_macro': 0.08907593485525409, 'f1_macro': 0.038507567793833415, 'mcc': -0.008808291092064496, 'roc_auc': nan, 'fit_time': 35.219536781311035, 'score_time': 0.13118314743041992}

FOLD 5/5


Shape train fold original: (703136, 64)
Shape val fold original  : (175784, 64)

Estrategia de rebalanceo: NearMiss_SMOTE
Distribución antes del rebalanceo:
LABEL
0     288000
1     221532
2     103246
3      61266
4       6100
5       5353
6       4388
7       3807
8       3524
9       3279
10      1749
11       869
12        15
13         8
Name: count, dtype: int64



Distribución después del undersampling:
LABEL
0     10000
1     10000
2     10000
3     10000
4      6100
5      5353
6      4388
7      3807
8      3524
9      3279
10     1749
11      869
12       15
13        8
Name: count, dtype: int64



SMOTE aplicado con k_neighbors=5
Distribución después de SMOTE:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
Name: count, dtype: int64

No se ha aplicado ENN.

Distribución final después del rebalanceo:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
Name: count, dtype: int64
Shape final: (140000, 64)



No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 5, 'train_original_rows': 703136, 'train_balanceado_rows': 140000, 'val_rows': 175784, 'accuracy': 0.08974081827697629, 'precision_weighted': 0.00984707652875737, 'recall_weighted': 0.08974081827697629, 'f1_weighted': 0.017715467895606626, 'precision_macro': 0.008857536147403826, 'recall_macro': 0.10607378356448018, 'f1_macro': 0.016055999224327478, 'mcc': 0.04229588009654718, 'roc_auc': nan, 'fit_time': 3.786428689956665, 'score_time': 0.1282806396484375}



In [11]:
df_folds = pd.DataFrame(resultados_folds)

df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,703136,140000,175784,0.087101,0.009742,0.087101,0.017524,0.007988,0.095205,0.014369,0.039047,NaN,17.942181,0.138570
1,2,703136,140000,175784,0.204302,0.081646,0.204302,0.116293,0.019350,0.080644,0.028004,-0.085834,NaN,36.375154,0.338807
2,3,703136,140000,175784,0.089809,0.009823,0.089809,0.017677,0.008873,0.107260,0.016092,0.042261,NaN,20.979352,0.128438
3,4,703136,140000,175784,0.315154,0.165160,0.315154,0.216468,0.029185,0.089076,0.038508,-0.008808,NaN,35.219537,0.131183
4,5,703136,140000,175784,0.089741,0.009847,0.089741,0.017715,0.008858,0.106074,0.016056,0.042296,NaN,3.786429,0.128281


In [12]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "modelo": "MLPClassifier",
        "mlp_hidden_layer_sizes": MLP_HIDDEN_LAYER_SIZES,
        "mlp_activation": MLP_ACTIVATION,
        "mlp_solver": MLP_SOLVER,
        "mlp_alpha": MLP_ALPHA,
        "mlp_batch_size": MLP_BATCH_SIZE,
        "mlp_learning_rate": MLP_LEARNING_RATE,
        "mlp_learning_rate_init": MLP_LEARNING_RATE_INIT,
        "mlp_max_iter": MLP_MAX_ITER,
        "mlp_early_stopping": MLP_EARLY_STOPPING,
        "mlp_validation_fraction": MLP_VALIDATION_FRACTION,
        "mlp_n_iter_no_change": MLP_N_ITER_NO_CHANGE,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv',
 'shape_train': {'rows': 878920, 'cols': 64},
 'parametros': {'modelo': 'MLPClassifier',
  'mlp_hidden_layer_sizes': (100,),
  'mlp_activation': 'relu',
  'mlp_solver': 'adam',
  'mlp_alpha': 0.0001,
  'mlp_batch_size': 'auto',
  'mlp_learning_rate': 'adaptive',
  'mlp_learning_rate_init': 0.001,
  'mlp_max_iter': 300,
  'mlp_early_stopping': True,
  'mlp_validation_fraction': 0.1,
  'mlp_n_iter_no_change': 10,
  'n_components_pca': 1,
  'estrategia_rebalanceo': 'NearMiss_SMOTE',
  'target_n': 10000,
  'nearmiss_version': 1,
  'smote_k_neighbors': 5,
  'enn_n_neighbors': 3},
 'metricas_media': {'accuracy': 0.15722136258134983,
  'precision_weighted': 0.055243637490697675,
  'recall_weighted': 0.15722136258134983,
  'f1_weighted': 0.0771356112198539,
  'precision_macro': 0.014850536724138368

In [13]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}\t"
    f"{summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.157221 ± 0.101457	0.055244 ± 0.068871	0.157221 ± 0.101457	0.077136 ± 0.088835	0.014851 ± 0.009280	0.095652 ± 0.011314	0.022606 ± 0.010431	0.005792 ± 0.055626	nan ± nan


In [14]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1__folds.csv


In [15]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1__summary_cv.json


In [16]:
df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,703136,140000,175784,0.087101,0.009742,0.087101,0.017524,0.007988,0.095205,0.014369,0.039047,NaN,17.942181,0.138570
1,2,703136,140000,175784,0.204302,0.081646,0.204302,0.116293,0.019350,0.080644,0.028004,-0.085834,NaN,36.375154,0.338807
2,3,703136,140000,175784,0.089809,0.009823,0.089809,0.017677,0.008873,0.107260,0.016092,0.042261,NaN,20.979352,0.128438
3,4,703136,140000,175784,0.315154,0.165160,0.315154,0.216468,0.029185,0.089076,0.038508,-0.008808,NaN,35.219537,0.131183
4,5,703136,140000,175784,0.089741,0.009847,0.089741,0.017715,0.008858,0.106074,0.016056,0.042296,NaN,3.786429,0.128281


In [17]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(219731, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,714,0,0.000082,2,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,1,8.202000e-05,1.499450e+09,0.000000,1.499450e+09,0.0,0.0,2
1,80,0,0.172038,12,396,7240,0,999.25,4.955239e+06,19201.959184,...,1,0,0,1.563983e-02,2.867301e-02,0.058456,4.013026e-02,0.0,0.0,1
2,3003,0,0.000043,2,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,1,4.292000e-05,1.499450e+09,0.000000,1.499450e+09,0.0,0.0,2
3,3389,0,0.000098,2,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,1,9.799000e-05,1.499450e+09,0.000000,1.499450e+09,0.0,0.0,2
4,80,0,0.000000,1,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,0,1.499344e+09,1.499344e+09,0.000000,0.000000e+00,0.0,0.0,10


In [18]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,90000
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


In [19]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (219731, 63)
Shape y_test: (219731,)


In [20]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [21]:
print("Rebalanceando todo el train original para entrenar el modelo final...")

df_train_balanceado_final = rebalancear_train_fold(
    df_fold_train=df_train,
    label_col=LABEL_COL,
    target_n=TARGET_N,
    random_state=RANDOM_STATE,
    nearmiss_version=NEARMISS_VERSION,
    smote_k_neighbors=SMOTE_K_NEIGHBORS,
    estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
    enn_n_neighbors=ENN_N_NEIGHBORS
)

X_train_balanceado_final = df_train_balanceado_final.drop(columns=[LABEL_COL])
y_train_balanceado_final = df_train_balanceado_final[LABEL_COL]

pipeline.fit(X_train_balanceado_final, y_train_balanceado_final)

print("Modelo final entrenado con todo el train rebalanceado.")
print("Train original   :", df_train.shape)
print("Train balanceado :", df_train_balanceado_final.shape)

Rebalanceando todo el train original para entrenar el modelo final...


Estrategia de rebalanceo: NearMiss_SMOTE
Distribución antes del rebalanceo:
LABEL
0     360000
1     276914
2     129058
3      76583
4       7625
5       6691
6       5485
7       4759
8       4406
9       4098
10      2186
11      1086
12        19
13        10
Name: count, dtype: int64



Distribución después del undersampling:
LABEL
0     10000
1     10000
2     10000
3     10000
4      7625
5      6691
6      5485
7      4759
8      4406
9      4098
10     2186
11     1086
12       19
13       10
Name: count, dtype: int64



SMOTE aplicado con k_neighbors=5
Distribución después de SMOTE:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
Name: count, dtype: int64

No se ha aplicado ENN.

Distribución final después del rebalanceo:
LABEL
0     10000
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
13    10000
Name: count, dtype: int64
Shape final: (140000, 64)



Modelo final entrenado con todo el train rebalanceado.
Train original   : (878920, 64)
Train balanceado : (140000, 64)


In [22]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 219731


In [23]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.0883307316673569,
 'precision_weighted': 0.009766989064957474,
 'recall_weighted': 0.0883307316673569,
 'f1_weighted': 0.017569101722368127,
 'precision_macro': 0.008396331145273333,
 'recall_macro': 0.10716711030307256,
 'f1_macro': 0.015173295793805618,
 'roc_auc': 0.5109151688133874,
 'mcc': 0.04054385520017741}

In [24]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{metricas_test['accuracy']:.6f}\t"
    f"{metricas_test['precision_weighted']:.6f}\t"
    f"{metricas_test['recall_weighted']:.6f}\t"
    f"{metricas_test['f1_weighted']:.6f}\t"
    f"{metricas_test['precision_macro']:.6f}\t"
    f"{metricas_test['recall_macro']:.6f}\t"
    f"{metricas_test['f1_macro']:.6f}\t"
    f"{metricas_test['mcc']:.6f}\t"
    f"{metricas_test['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.088331	0.009767	0.088331	0.017569	0.008396	0.107167	0.015173	0.040544	0.510915


In [25]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0,0,0,69066,0,0,0,0,0,0,20934,0,0,0
1,0,0,0,44234,0,0,0,0,0,0,24995,0,0,0
2,0,0,0,31841,0,0,0,0,0,0,424,0,0,0
3,0,0,0,19135,0,0,0,0,0,0,11,0,0,0
4,0,0,0,1272,0,0,0,0,0,0,634,0,0,0
5,0,0,0,1579,0,0,0,0,0,0,94,0,0,0
6,0,0,0,1014,0,0,0,0,0,0,357,0,0,0
7,0,0,0,620,0,0,0,0,0,0,570,0,0,0
8,0,0,0,955,0,0,0,0,0,0,147,0,0,0
9,0,0,0,833,0,0,0,0,0,0,191,0,0,0


In [26]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     90000
           1       0.00      0.00      0.00     69229
           2       0.00      0.00      0.00     32265
           3       0.11      1.00      0.20     19146
           4       0.00      0.00      0.00      1906
           5       0.00      0.00      0.00      1673
           6       0.00      0.00      0.00      1371
           7       0.00      0.00      0.00      1190
           8       0.00      0.00      0.00      1102
           9       0.00      0.00      0.00      1024
          10       0.01      0.50      0.01       547
          11       0.00      0.00      0.00       271
          12       0.00      0.00      0.00         5
          13       0.00      0.00      0.00         2

    accuracy                           0.09    219731
   macro avg       0.01      0.11      0.02    219731
weighted avg       0.01      0.

In [27]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "modelo": "MLPClassifier",
        "mlp_hidden_layer_sizes": MLP_HIDDEN_LAYER_SIZES,
        "mlp_activation": MLP_ACTIVATION,
        "mlp_solver": MLP_SOLVER,
        "mlp_alpha": MLP_ALPHA,
        "mlp_batch_size": MLP_BATCH_SIZE,
        "mlp_learning_rate": MLP_LEARNING_RATE,
        "mlp_learning_rate_init": MLP_LEARNING_RATE_INIT,
        "mlp_max_iter": MLP_MAX_ITER,
        "mlp_early_stopping": MLP_EARLY_STOPPING,
        "mlp_validation_fraction": MLP_VALIDATION_FRACTION,
        "mlp_n_iter_no_change": MLP_N_ITER_NO_CHANGE,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv',
 'shape_test': {'rows': 219731, 'cols': 64},
 'parametros': {'modelo': 'MLPClassifier',
  'mlp_hidden_layer_sizes': (100,),
  'mlp_activation': 'relu',
  'mlp_solver': 'adam',
  'mlp_alpha': 0.0001,
  'mlp_batch_size': 'auto',
  'mlp_learning_rate': 'adaptive',
  'mlp_learning_rate_init': 0.001,
  'mlp_max_iter': 300,
  'mlp_early_stopping': True,
  'mlp_validation_fraction': 0.1,
  'mlp_n_iter_no_change': 10,
  'n_components_pca': 1,
  'estrategia_rebalanceo': 'NearMiss_SMOTE',
  'target_n': 10000,
  'nearmiss_version': 1,
  'smote_k_neighbors': 5,
  'enn_n_neighbors': 3},
 'metricas_test': {'accuracy': 0.0883307316673569,
  'precision_weighted': 0.009766989064957474,
  'recall_weighted': 0.0883307316673569,
  'f1_weighted': 0.017569101722368127,
  'precision_macro': 0.008396331145273333,
  

In [28]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1__metricas_test.csv


In [29]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1__confusion_matrix_test.csv


In [30]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1/BCCC17__split__v1__NearMiss_SMOTE_pca4_mlp__v1__summary_test.json


In [31]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.15722136258134983, 'precision_weighted': 0.055243637490697675, 'recall_weighted': 0.15722136258134983, 'f1_weighted': 0.0771356112198539, 'precision_macro': 0.014850536724138368, 'recall_macro': 0.09565181680935604, 'f1_macro': 0.022605598167705767, 'mcc': 0.005792192152260714, 'roc_auc': nan, 'fit_time': 22.86053047180176, 'score_time': 0.17305569648742675}

TEST:
{'accuracy': 0.0883307316673569, 'precision_weighted': 0.009766989064957474, 'recall_weighted': 0.0883307316673569, 'f1_weighted': 0.017569101722368127, 'precision_macro': 0.008396331145273333, 'recall_macro': 0.10716711030307256, 'f1_macro': 0.015173295793805618, 'mcc': 0.04054385520017741}
